# Workshop: Building a Retrieval-Augmented Generator (RAG) with LangChain and Pinecone

## Parte 1: RAG Básico con LangChain y OpenAI
En esta primera sección, implementaremos un flujo RAG local utilizando:
- **Carga de documentos:** Lectura de nuestra base de conocimiento local (`data/knowledge.txt`).
- **Text Splitting:** División del texto en fragmentos (chunks) manejables.
- **Embeddings y Vector Store local:** Conversión de texto a vectores usando OpenAI y almacenamiento temporal (FAISS).
- **Retrieval & Generation:** Búsqueda de contexto relevante y generación de respuestas con el modelo de lenguaje de OpenAI.

In [1]:
import os
from dotenv import load_dotenv

# Cargar las variables del archivo .env
load_dotenv()

# Verificar que las API keys estén cargadas correctamente (no imprimiremos las claves por seguridad)
openai_key = os.getenv("OPENAI_API_KEY")
pinecone_key = os.getenv("PINECONE_API_KEY")

if openai_key:
    print("¡OpenAI API Key cargada con éxito!")
else:
    print("Error: No se encontró la OPENAI_API_KEY en el archivo .env")

if pinecone_key:
    print("¡Pinecone API Key cargada con éxito!")
else:
    print("Error: No se encontró la PINECONE_API_KEY en el archivo .env")

¡OpenAI API Key cargada con éxito!
¡Pinecone API Key cargada con éxito!


In [2]:
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

# 1. Cargar el documento de conocimiento local
loader = TextLoader("data/knowledge.txt", encoding="utf-8")
documents = loader.load()

# 2. Dividir el texto en fragmentos (chunks) manejables
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50
)
docs = text_splitter.split_documents(documents)

print(f"Documento cargado y dividido en {len(docs)} fragmento(s).")
for i, doc in enumerate(docs):
    print(f"\n--- Fragmento {i+1} ---")
    print(doc.page_content)

C:\Users\jimmy\AppData\Local\Temp\ipykernel_13536\1695584216.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import TextLoader


Documento cargado y dividido en 1 fragmento(s).

--- Fragmento 1 ---
LangChain es un framework para desarrollar aplicaciones impulsadas por modelos de lenguaje.
Pinecone es una base de datos vectorial en la nube diseñada para búsquedas de similitud a gran escala y alta velocidad.


In [3]:
# Sobrescribir el archivo de conocimiento con codificación UTF-8 estándar
contenido = """LangChain es un framework para desarrollar aplicaciones impulsadas por modelos de lenguaje.
Pinecone es una base de datos vectorial en la nube diseñada para búsquedas de similitud a gran escala y alta velocidad."""

with open("data/knowledge.txt", "w", encoding="utf-8") as f:
    f.write(contenido)

print("¡Archivo knowledge.txt regenerado exitosamente con codificación UTF-8!")

¡Archivo knowledge.txt regenerado exitosamente con codificación UTF-8!


In [5]:
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_community.vectorstores import FAISS
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

# 1. Inicializar embeddings y base de datos vectorial local (FAISS)
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")
vectorstore = FAISS.from_documents(docs, embeddings)
retriever = vectorstore.as_retriever(search_kwargs={"k": 2})

# 2. Configurar el modelo de lenguaje y el prompt del sistema
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

prompt_template = """Responde a la pregunta basándote exclusivamente en el siguiente contexto:

{context}

Pregunta: {question}"""

prompt = ChatPromptTemplate.from_template(prompt_template)

# Función para formatear los documentos recuperados en un solo texto
def format_docs(documents):
    return "\n\n".join(doc.page_content for doc in documents)

# 3. Construir la cadena RAG con LCEL
rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

# 4. Realizar la consulta de prueba
query = "¿Qué es Pinecone?"
response = rag_chain.invoke(query)

print(f"Pregunta: {query}")
print(f"\nRespuesta del RAG:\n{response}")

Pregunta: ¿Qué es Pinecone?

Respuesta del RAG:
Pinecone es una base de datos vectorial en la nube diseñada para búsquedas de similitud a gran escala y alta velocidad.


In [6]:
!pip install langchain-classic langchain-core


[notice] A new release of pip is available: 25.3 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


## Parte 2: Enhancing the RAG with Pinecone Vector Database
En esta segunda sección, conectamos nuestro RAG a **Pinecone**, una base de datos vectorial en la nube diseñada para almacenar e indexar embeddings a gran escala.

### Ventajas de utilizar Pinecone frente al almacenamiento en memoria (FAISS):
1. **Persistencia y Escalabilidad:** Los datos no residen únicamente en la memoria RAM local; se almacenan en la nube y pueden escalar a miles de millones de vectores.
2. **Consultas eficientes y de baja latencia:** Indexación vectorial optimizada para búsquedas en tiempo real.
3. **Mantenimiento y Filtrado:** Permite aplicar filtros por metadatos y actualizar documentos sin reentrenar ni reindexar todo desde cero.

In [1]:
import os
from pinecone import Pinecone, ServerlessSpec
from langchain_pinecone import PineconeVectorStore

# 1. Configurar e Inicializar cliente de Pinecone
pinecone_api_key = os.getenv("PINECONE_API_KEY")
pc = Pinecone(api_key=pinecone_api_key)

index_name = "rag-workshop"

# Verificar o crear el índice automáticamente
if index_name not in [index.name for index in pc.list_indexes()]:
    pc.create_index(
        name=index_name,
        dimension=1536, # Coincide con text-embedding-3-small
        metric="cosine",
        spec=ServerlessSpec(cloud="aws", region="us-east-1")
    )
    print(f"Índice '{index_name}' creado exitosamente en Pinecone.")
else:
    print(f"Conectado al índice existente '{index_name}' en Pinecone.")

# 2. Cargar y almacenar los embeddings en Pinecone
pinecone_vectorstore = PineconeVectorStore.from_documents(
    documents=docs,
    embedding=embeddings,
    index_name=index_name
)

# 3. Crear el Retriever desde Pinecone
pinecone_retriever = pinecone_vectorstore.as_retriever(search_kwargs={"k": 2})

# 4. Construir la cadena RAG impulsada por Pinecone
pinecone_rag_chain = (
    {"context": pinecone_retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

# 5. Probar la consulta interactuando con Pinecone
query = "¿Qué ventajas ofrece una base de datos vectorial como Pinecone?"
response = pinecone_rag_chain.invoke(query)

print(f"Pregunta: {query}")
print(f"\nRespuesta del RAG con Pinecone:\n{response}")

PineconeConfigurationError: You haven't specified an API key. Please either set the PINECONE_API_KEY environment variable or pass the 'api_key' keyword argument to the Pinecone client constructor.

In [6]:
import os
from dotenv import load_dotenv
from pinecone import Pinecone, ServerlessSpec
from langchain_pinecone import PineconeVectorStore

# Cargar obligatoriamente las variables de entorno
load_dotenv()

# Obtener y verificar la clave de Pinecone
pinecone_api_key = os.getenv("PINECONE_API_KEY")

if not pinecone_api_key:
    raise ValueError("¡La API Key de Pinecone no está configurada en el archivo .env!")

# Inicializar cliente de Pinecone explícitamente
pc = Pinecone(api_key=pinecone_api_key)

index_name = "rag-workshop"

# Verificar o crear el índice automáticamente
if index_name not in [index.name for index in pc.list_indexes()]:
    pc.create_index(
        name=index_name,
        dimension=1536, # Coincide con text-embedding-3-small de OpenAI
        metric="cosine",
        spec=ServerlessSpec(cloud="aws", region="us-east-1")
    )
    print(f"Índice '{index_name}' creado exitosamente en Pinecone.")
else:
    print(f"Conectado al índice existente '{index_name}' en Pinecone.")

# 2. Cargar y almacenar los embeddings en Pinecone
pinecone_vectorstore = PineconeVectorStore.from_documents(
    documents=docs,
    embedding=embeddings,
    index_name=index_name
)

# 3. Crear el Retriever desde Pinecone
pinecone_retriever = pinecone_vectorstore.as_retriever(search_kwargs={"k": 2})

# 4. Construir la cadena RAG impulsada por Pinecone
pinecone_rag_chain = (
    {"context": pinecone_retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

# 5. Probar la consulta interactuando con Pinecone
query = "¿Qué ventajas ofrece una base de datos vectorial como Pinecone?"
response = pinecone_rag_chain.invoke(query)

print(f"Pregunta: {query}")
print(f"\nRespuesta del RAG con Pinecone:\n{response}")

Conectado al índice existente 'rag-workshop' en Pinecone.


NameError: name 'format_docs' is not defined

In [3]:
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Recargar y dividir el documento para que 'docs' vuelva a estar definido
loader = TextLoader("data/knowledge.txt", encoding="utf-8")
documents = loader.load()

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50
)
docs = text_splitter.split_documents(documents)
print("¡Variable 'docs' recargada con éxito!")

¡Variable 'docs' recargada con éxito!


C:\Users\jimmy\AppData\Local\Temp\ipykernel_8228\2374986853.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import TextLoader


In [5]:
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

# 1. Recargar y dividir el documento ('docs')
loader = TextLoader("data/knowledge.txt", encoding="utf-8")
documents = loader.load()

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50
)
docs = text_splitter.split_documents(documents)

# 2. Inicializar los embeddings de OpenAI ('embeddings')
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

print("¡Variables 'docs' y 'embeddings' definidas correctamente en memoria!")

¡Variables 'docs' y 'embeddings' definidas correctamente en memoria!


In [7]:
import os
from dotenv import load_dotenv
from pinecone import Pinecone, ServerlessSpec
from langchain_pinecone import PineconeVectorStore
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

# 1. Cargar entorno y credenciales
load_dotenv()
pinecone_api_key = os.getenv("PINECONE_API_KEY")

if not pinecone_api_key:
    raise ValueError("¡La API Key de Pinecone no está configurada en el archivo .env!")

pc = Pinecone(api_key=pinecone_api_key)
index_name = "rag-workshop"

# 2. Verificar o crear el índice en Pinecone
if index_name not in [index.name for index in pc.list_indexes()]:
    pc.create_index(
        name=index_name,
        dimension=1536,
        metric="cosine",
        spec=ServerlessSpec(cloud="aws", region="us-east-1")
    )
    print(f"Índice '{index_name}' creado exitosamente en Pinecone.")
else:
    print(f"Conectado al índice existente '{index_name}' en Pinecone.")

# 3. Cargar el documento y generar embeddings
loader = TextLoader("data/knowledge.txt", encoding="utf-8")
documents = loader.load()

text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
docs = text_splitter.split_documents(documents)

embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

# 4. Almacenar en Pinecone
pinecone_vectorstore = PineconeVectorStore.from_documents(
    documents=docs,
    embedding=embeddings,
    index_name=index_name
)

pinecone_retriever = pinecone_vectorstore.as_retriever(search_kwargs={"k": 2})

# 5. Configurar modelo de lenguaje y prompt
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

prompt_template = """Responde a la pregunta basándote exclusivamente en el siguiente contexto:

{context}

Pregunta: {question}"""

prompt = ChatPromptTemplate.from_template(prompt_template)

# Función para formatear documentos
def format_docs(documents):
    return "\n\n".join(doc.page_content for doc in documents)

# 6. Construir la cadena RAG con LCEL y ejecutar
pinecone_rag_chain = (
    {"context": pinecone_retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

query = "¿Qué ventajas ofrece una base de datos vectorial como Pinecone?"
response = pinecone_rag_chain.invoke(query)

print(f"\nPregunta: {query}")
print(f"\nRespuesta del RAG con Pinecone:\n{response}")

Conectado al índice existente 'rag-workshop' en Pinecone.

Pregunta: ¿Qué ventajas ofrece una base de datos vectorial como Pinecone?

Respuesta del RAG con Pinecone:
Una base de datos vectorial como Pinecone ofrece ventajas como búsquedas de similitud a gran escala y alta velocidad, lo que permite manejar grandes volúmenes de datos de manera eficiente y rápida. Esto es especialmente útil para aplicaciones que requieren encontrar relaciones o similitudes entre datos de forma efectiva.
